# Day 15: Document-Specific Q&A

## Goal

Tune prompts for invoice-style document questions and produce structured answers.

By the end of this notebook you will have:

- an invoice-like document sample
- focused prompts for `vendor`, `total amount`, and `items`
- a structured extraction schema
- a model output shaped for downstream automation

This is the step where general RAG becomes task-specific document extraction.

## Step 0: Sync the Project Once

The Day 15 packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Set the values you want in `.env`.

Example:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=your_openai_model
OLLAMA_MODEL=llama3.2
OLLAMA_BASE_URL=http://localhost:11434
```

This notebook supports either OpenAI or Ollama for the extraction prompts.

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

print("Imports loaded successfully.")


In [ ]:
load_dotenv(override=True)

config_summary = {
    "OPENAI_API_KEY_present": bool(os.getenv("OPENAI_API_KEY")),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),
    "OLLAMA_BASE_URL": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
}

config_summary

## Step 1: Create a Sample Invoice Document

For Day 15, the document needs to be invoice-specific so prompt tuning is easy to understand.

Later, you can replace this text with extracted PDF or OCR output from earlier days.

In [ ]:
invoice_text = """
Invoice Number: INV-2026-041
Invoice Date: 2026-03-21
Vendor: ACME Supplies Pvt Ltd
Bill To: Orion Manufacturing Ltd

Items:
1. Printer Ink Cartridge - Qty 4 - Unit Price 1800 INR - Line Total 7200 INR
2. A4 Paper Ream - Qty 10 - Unit Price 320 INR - Line Total 3200 INR
3. USB Keyboard - Qty 2 - Unit Price 950 INR - Line Total 1900 INR

Subtotal: 12300 INR
Tax: 2214 INR
Total Amount: 14514 INR
Payment Due: 2026-04-05
""".strip()

print(invoice_text)

## Step 2: Create a Chat Model Helper

This keeps the rest of the notebook identical whether you choose OpenAI or Ollama.

In [ ]:
def get_chat_model(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_MODEL")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        if not model:
            raise ValueError("OPENAI_MODEL is not set in .env")
        return ChatOpenAI(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL")
        if not model:
            raise ValueError("OLLAMA_MODEL is not set in .env")
        return ChatOllama(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


provider = "openai"  # Change to "ollama" if you want to use Ollama.
llm = get_chat_model(provider)
print("Model ready.")

## Step 3: Ask a Generic Question

A broad prompt can work, but the answer is less predictable for automation.

This is the baseline before prompt tuning.

In [ ]:
generic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant reading business documents."),
        ("human", """Read the invoice text and summarize the important details.

{document_text}"""),
    ]
)


def summarize_invoice(text: str) -> str:
    messages = generic_prompt.format_messages(document_text=text)
    response = llm.invoke(messages)
    return response.content


print(summarize_invoice(invoice_text))


## Step 4: Tune the Prompt for Specific Fields

Now we ask targeted questions so the model focuses on the exact fields we care about.

In [ ]:
field_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You extract one field from invoice text. Answer only with the requested value. If the value is missing, answer NOT_FOUND.",
        ),
        (
            "human",
            """Field to extract: {field_name}

Invoice text:
{document_text}""",
        ),
    ]
)


def extract_field(field_name: str, document_text: str) -> str:
    messages = field_prompt.format_messages(field_name=field_name, document_text=document_text)
    response = llm.invoke(messages)
    return response.content.strip()


print("Field extraction helper ready.")


In [ ]:
vendor_result = extract_field("vendor", invoice_text)
print(vendor_result)


In [ ]:
total_amount_result = extract_field("total amount", invoice_text)
print(total_amount_result)


In [ ]:
items_result = extract_field("items", invoice_text)
print(items_result)


## Step 5: Define a Structured JSON Shape

This is the format we want the model to produce.

The structure matters because later systems can store or validate these values automatically.


In [ ]:
expected_schema = {
    "vendor": "string or null",
    "invoice_number": "string or null",
    "invoice_date": "string or null",
    "total_amount": "string or null",
    "items": ["string", "..."],
    "notes": "string or null",
}

expected_schema


## Step 6: Prompt the Model for Structured Extraction

The prompt is now tuned for a very specific business task.

Important guardrails:
- do not invent values
- use only what is present in the document
- return a JSON object with the expected keys


In [ ]:
structured_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an invoice extraction assistant. Use only the provided document text. Do not infer missing values.",
        ),
        (
            "human",
            """Extract the vendor, invoice number, invoice date, total amount, and item names from the invoice text below.

Return a JSON object with keys: vendor, invoice_number, invoice_date, total_amount, items, notes.

Invoice text:
{document_text}""",
        ),
    ]
)


def extract_structured(document_text: str) -> dict:
    messages = structured_prompt.format_messages(document_text=document_text)
    response = llm.invoke(messages)
    raw = response.content.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw}


print("Structured extraction helper ready.")


In [ ]:
structured_result = extract_structured(invoice_text)
print(structured_result)


In [ ]:
print(structured_result)


## Step 7: Try Another Document Variation

Real documents are messy.

Change the text below to simulate missing fields, different labels, or noisy OCR output, then run the structured extraction cell again.

In [ ]:
invoice_text_variant = """
Supplier Name: Bright Office Retail
Doc No: BR-8821
Date: 2026/03/19
Products Purchased:
- Whiteboard Marker Set
- Desk Organizer Tray
Amount Payable: 3890 INR
""".strip()

invoice_text_variant

In [ ]:
print(extract_structured(invoice_text_variant))


## Day 15 Recap

You now have a document-specific Q&A and extraction flow:

- a broad prompt for general summary
- tuned prompts for specific invoice fields
- a structured schema for downstream use
- model output shaped as machine-friendly data

This is the practical step from generic RAG into invoice-style extraction workflows.